# Notebook 01 — Parquet Ingestion & EDA

**Source:** Nigerian Mobile Money transactions (`raw_mobile_money.parquet`)  
**Owner:** Nadine + Patrick (original ingestion) | Consolidated by Elie Khairallah  
**Rubric:** §4.1 Acquisition, §4.3 Cleaning, §4.4 Processing

This notebook loads the raw parquet file, validates its structure, applies cleaning transformations, and produces exploratory visualisations that inform the pipeline design decisions.

In [ ]:
import sys
import os
from pathlib import Path
# Ensure project root is on path when notebook runs from notebooks/
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))
Path('data/audit').mkdir(exist_ok=True)
print('Working directory:', os.getcwd())

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — safe for Restart→Run All
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded')

## 1. Ingestion

Load the raw parquet file using `src/ingestion/ingest_parquet.py`. The module validates column completeness and wallet_id format on load.

In [ ]:
from src.ingestion import ingest_parquet
df_raw = ingest_parquet.load_raw()
print(f'Shape: {df_raw.shape}')
print(df_raw.dtypes)

In [ ]:
print('Null counts:')
print(df_raw.isnull().sum())
print(f'\nDuplicate rows: {df_raw.duplicated().sum()}')
print(f'Duplicate transaction_ids: {df_raw["transaction_id"].duplicated().sum()}')

**Finding:** Zero nulls, zero duplicates. All 4 million rows are structurally clean on ingest.

In [ ]:
print('Wallet ID format check (first 5):', df_raw['wallet_id'].head().tolist())
print('Timestamp range:', df_raw['timestamp'].min(), '->', df_raw['timestamp'].max())

## 2. Cleaning

Apply cleaning per §6.5 rules: type coercions, agent_id normalisation, IQR outlier flagging.

In [ ]:
from src.cleaning import clean_parquet
df_clean, report = clean_parquet.clean(df_raw)
print('Cleaning actions:')
for a in report['actions']:
    print(f'  • {a}')
print(f'\nRows before: {report["before"]["row_count"]:,}')
print(f'Rows after:  {report["after"]["row_count"]:,}')

In [ ]:
# Show outlier flag counts
for col in ['amount_ngn', 'fee_ngn', 'balance_after_ngn']:
    flag = f'is_{col}_outlier'
    n = df_clean[flag].sum()
    print(f'{flag}: {n:,} rows ({n/len(df_clean)*100:.2f}%)')

## 3. EDA

### 3.1 Transaction Type Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
types = df_clean['transaction_type'].value_counts()
types.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Transaction Volume by Type', fontweight='bold')
axes[0].set_xlabel('Type'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
for bar in axes[0].patches:
    axes[0].annotate(f'{int(bar.get_height()):,}', (bar.get_x()+bar.get_width()/2, bar.get_height()), ha='center', va='bottom', fontsize=8)

channels = df_clean['channel'].value_counts()
channels.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['#4C72B0','#DD8452','#55A868','#C44E52'])
axes[1].set_title('Channel Mix', fontweight='bold')
axes[1].set_ylabel('')
plt.tight_layout()
fig.savefig('data/audit/fig_01_tx_types.png', dpi=100, bbox_inches='tight')
plt.show()
print('Fig saved: data/audit/fig_01_tx_types.png')

**Interpretation:** Airtime top-ups dominate (25%), followed by P2P sends/receives (~40% combined). USSD channel captures 50% of volume, reflecting the financial-inclusion mandate of the platform — many users lack smartphones.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KYC tier
kyc = df_clean['kyc_tier'].value_counts()
kyc.plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452','#55A868'], edgecolor='black')
axes[0].set_title('Customers by KYC Tier', fontweight='bold')
axes[0].set_xlabel('KYC Tier'); axes[0].set_ylabel('Transaction Count')
axes[0].tick_params(axis='x', rotation=0)

# Churn vs fraud
cf = pd.DataFrame({'Rate (%)': [df_clean['churn_30d'].mean()*100, df_clean['fraud_flag'].mean()*100]},
                  index=['Churn (30d)', 'Fraud'])
cf.plot(kind='bar', ax=axes[1], color=['#C44E52','#8172B2'], edgecolor='black', legend=False)
axes[1].set_title('Churn & Fraud Rates', fontweight='bold')
axes[1].set_ylabel('Rate (%)')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
fig.savefig('data/audit/fig_02_kyc_churn_fraud.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretation:** Tier 1 accounts represent 58% of transactions — entry-level KYC. Churn rate is 6%; fraud is 1.5%. Both are rare enough that the stratified sampling strategy is essential to preserve these distributions in the 500K sample.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
monthly = df_clean.groupby(df_clean['timestamp'].dt.to_period('M')).agg(
    tx_count=('transaction_id','count'), volume=('amount_ngn','sum')).reset_index()
monthly['timestamp'] = monthly['timestamp'].astype(str)
ax2 = ax.twinx()
ax.bar(monthly['timestamp'], monthly['tx_count'], color='steelblue', alpha=0.7, label='Tx count')
ax2.plot(monthly['timestamp'], monthly['volume']/1e9, color='red', marker='o', label='Volume (₦B)')
ax.set_title('Monthly Transaction Count & Volume', fontweight='bold')
ax.set_xlabel('Month'); ax.set_ylabel('Transaction Count'); ax2.set_ylabel('Volume (₦ Billions)')
ax.tick_params(axis='x', rotation=30)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
fig.savefig('data/audit/fig_03_monthly_volume.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Save Processed Outputs

In [ ]:
from pathlib import Path
Path('data/processed').mkdir(exist_ok=True)
Path('data/samples').mkdir(exist_ok=True)

# Save cleaned parquet
df_clean.to_parquet('data/processed/parquet_cleaned.parquet', index=False)
print('Saved: data/processed/parquet_cleaned.parquet')

# Save 1K sample for git
df_clean.sample(1000, random_state=42).to_csv('data/samples/parquet_sample_1k.csv', index=False)
print('Saved: data/samples/parquet_sample_1k.csv')
print(f'\nFinal shape: {df_clean.shape}')